# 11 — v11: H3 Geospatial Cell Encoding

**Current best:** v10 (postal-sector OOF smoothing) → Kaggle RMSE **21,755.56** | Goal: improve geographic price signal using Uber's H3 hexagonal grid.

## What changes vs v10

| # | Change | Why |
|---|---|---|
| 1 | **`h3_r8_median_price`** — OOF median at H3 resolution 8 (~0.74 km² cells, ~990 cells in SG) | Equal-area hexagonal zones capture price gradients better than postal sector boundaries |
| 2 | **`h3_r7_median_price`** — OOF median at H3 resolution 7 (~5.16 km² cells, ~140 cells in SG) | District-level spatial signal, coarser than r8 but more stable medians |

All other settings (model params, features, stacking architecture) are identical to v10.

## Why H3 over postal_sector?
- Postal sectors follow postal code geography — unequal geographic sizes, not aligned to price gradients
- H3 hexagonal cells are equal-area, geographically consistent tiles
- Two resolutions capture both neighbourhood-level (r8) and district-level (r7) price patterns
- Added **on top of** `postal_sector_median_price` — model decides which is more useful

---
## 1. Imports & Load Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from math import radians, cos, sin, asin, sqrt
import h3

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'H3 version: {h3.__version__}')
print(f'Test H3 cell (Raffles Place): {h3.latlng_to_cell(1.2847, 103.8510, 8)}')

Train: (150634, 77)  |  Test: (16737, 76)
H3 version: 4.4.2
Test H3 cell (Raffles Place): 886520db33fffff


---
## 2. Feature Engineering

Same as v10, plus two new H3 cell ID columns:
- **`h3_r7`** — H3 cell ID at resolution 7 (~5.16 km² per cell, district-level)
- **`h3_r8`** — H3 cell ID at resolution 8 (~0.74 km² per cell, neighbourhood-level)

These string cell IDs are used only as grouping keys for OOF encoding. They are dropped before model training.

In [3]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
CBD_LAT, CBD_LON = 1.2847, 103.8510

STREET_FREQ = train['street_name'].value_counts().to_dict()

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    df = df.copy()
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)

    # Existing features (same as v10)
    df['remaining_lease']  = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['dist_to_cbd']      = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate'] = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    df['street_freq']     = df['street_name'].map(STREET_FREQ).fillna(0)
    df['postal_sector'] = df['postal'].astype(str).str[:4]
    df['block_num']     = pd.to_numeric(
        df['block'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0)

    # NEW: H3 geospatial cell IDs at two resolutions
    df['h3_r7'] = df.apply(
        lambda r: h3.latlng_to_cell(r['Latitude'], r['Longitude'], 7), axis=1
    )
    df['h3_r8'] = df.apply(
        lambda r: h3.latlng_to_cell(r['Latitude'], r['Longitude'], 8), axis=1
    )

    return df, new_caps

train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')
print(f'postal_sector unique (train): {train_fe["postal_sector"].nunique()}')
print(f'h3_r7 unique (train): {train_fe["h3_r7"].nunique()}  |  test: {test_fe["h3_r7"].nunique()}')
print(f'h3_r8 unique (train): {train_fe["h3_r8"].nunique()}  |  test: {test_fe["h3_r8"].nunique()}')

Train: (150634, 96)  |  Test: (16737, 95)
postal_sector unique (train): 598
h3_r7 unique (train): 70  |  test: 70
h3_r8 unique (train): 259  |  test: 254


---
## 3. Per-Fold OOF Target Encoding

Same pattern as v10, extended with two H3 encoding columns:
- `h3_r8_median_price` — neighbourhood-level price signal (min_count=10)
- `h3_r7_median_price` — district-level price signal (min_count=5, coarser so more stable)

> **No leakage**: each row's encoded value is computed from the other 4 folds only.

In [4]:
MIN_COUNT_SECTOR = 10
MIN_COUNT_H3_R8  = 10
MIN_COUNT_H3_R7  = 5

def oof_group_median(group_series, y_series, n_splits=5, random_state=42, min_count=1):
    """OOF median target encoding. Groups with < min_count rows fall back to global median."""
    groups = group_series.values
    y      = y_series.values
    encoded    = np.zeros(len(groups))
    global_med = np.median(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(groups):
        fold_counts = {}
        fold_vals   = {}
        for g, p in zip(groups[tr_idx], y[tr_idx]):
            fold_counts[g] = fold_counts.get(g, 0) + 1
            fold_vals.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_vals.items()
                    if fold_counts[g] >= min_count}
        for i in va_idx:
            encoded[i] = fold_map.get(groups[i], global_med)
    return encoded

# --- Training set: OOF encoding (no leakage) ---
train_fe['town_median_price']          = oof_group_median(train_fe['town'],          train['resale_price'])
train_fe['postal_sector_median_price'] = oof_group_median(train_fe['postal_sector'], train['resale_price'],
                                                           min_count=MIN_COUNT_SECTOR)
train_fe['flat_model_median_price']    = oof_group_median(train_fe['flat_model'],    train['resale_price'])
train_fe['h3_r8_median_price']         = oof_group_median(train_fe['h3_r8'],         train['resale_price'],
                                                           min_count=MIN_COUNT_H3_R8)
train_fe['h3_r7_median_price']         = oof_group_median(train_fe['h3_r7'],         train['resale_price'],
                                                           min_count=MIN_COUNT_H3_R7)

# --- Test set: full-training-set map (no leakage; test labels unknown) ---
_tmp = pd.DataFrame({
    'town':          train_fe['town'].values,
    'postal_sector': train_fe['postal_sector'].values,
    'flat_model':    train_fe['flat_model'].values,
    'h3_r7':         train_fe['h3_r7'].values,
    'h3_r8':         train_fe['h3_r8'].values,
    'resale_price':  train['resale_price'].values,
})
full_town_map   = _tmp.groupby('town')['resale_price'].median()
sector_counts   = _tmp.groupby('postal_sector')['resale_price'].count()
valid_sectors   = sector_counts[sector_counts >= MIN_COUNT_SECTOR].index
full_sector_map = _tmp[_tmp['postal_sector'].isin(valid_sectors)].groupby('postal_sector')['resale_price'].median()
full_model_map  = _tmp.groupby('flat_model')['resale_price'].median()

h3r8_counts     = _tmp.groupby('h3_r8')['resale_price'].count()
valid_h3r8      = h3r8_counts[h3r8_counts >= MIN_COUNT_H3_R8].index
full_h3r8_map   = _tmp[_tmp['h3_r8'].isin(valid_h3r8)].groupby('h3_r8')['resale_price'].median()

h3r7_counts     = _tmp.groupby('h3_r7')['resale_price'].count()
valid_h3r7      = h3r7_counts[h3r7_counts >= MIN_COUNT_H3_R7].index
full_h3r7_map   = _tmp[_tmp['h3_r7'].isin(valid_h3r7)].groupby('h3_r7')['resale_price'].median()

test_fe['town_median_price']          = test_fe['town'].map(full_town_map).fillna(full_town_map.median())
test_fe['postal_sector_median_price'] = test_fe['postal_sector'].map(full_sector_map).fillna(full_sector_map.median())
test_fe['flat_model_median_price']    = test_fe['flat_model'].map(full_model_map).fillna(full_model_map.median())
test_fe['h3_r8_median_price']         = test_fe['h3_r8'].map(full_h3r8_map).fillna(full_h3r8_map.median())
test_fe['h3_r7_median_price']         = test_fe['h3_r7'].map(full_h3r7_map).fillna(full_h3r7_map.median())

print('OOF encoding done.')
print(f'h3_r8 cells with >= {MIN_COUNT_H3_R8} rows: {len(valid_h3r8)} of {len(h3r8_counts)}')
print(f'h3_r7 cells with >= {MIN_COUNT_H3_R7} rows: {len(valid_h3r7)} of {len(h3r7_counts)}')
print(f'h3_r8_median_price nulls — train: {train_fe["h3_r8_median_price"].isna().sum()}, test: {test_fe["h3_r8_median_price"].isna().sum()}')
print(f'h3_r7_median_price nulls — train: {train_fe["h3_r7_median_price"].isna().sum()}, test: {test_fe["h3_r7_median_price"].isna().sum()}')

OOF encoding done.
h3_r8 cells with >= 10 rows: 254 of 259
h3_r7 cells with >= 5 rows: 70 of 70
h3_r8_median_price nulls — train: 0, test: 0
h3_r7_median_price nulls — train: 0, test: 0


---
## 4. Prepare X and y

`h3_r7` and `h3_r8` (raw string cell IDs) are dropped — replaced by their OOF-encoded median price columns.

In [5]:
DROP_ALL = (['id', 'resale_price', 'postal', 'block', 'h3_r7', 'h3_r8']
            + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms'])

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')
print(f'New H3 features present: h3_r8_median_price={"h3_r8_median_price" in num_cols}, h3_r7_median_price={"h3_r7_median_price" in num_cols}')
print(f'H3 raw strings dropped: h3_r7={"h3_r7" not in X.columns}, h3_r8={"h3_r8" not in X.columns}')
assert X_test.shape[1] == X.shape[1], 'X/X_test column mismatch'

Features: 73  (num=58, cat=15)
New H3 features present: h3_r8_median_price=True, h3_r7_median_price=True
H3 raw strings dropped: h3_r7=True, h3_r8=True


---
## 5. Preprocessing Pipeline

In [6]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def neg_dollar_rmse(y_log_true, y_log_pred):
    return -np.sqrt(mean_squared_error(np.expm1(y_log_true), np.expm1(y_log_pred)))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)
print('Preprocessor ready.')

Preprocessor ready.


---
## 6. RandomizedSearchCV — XGBoost

Same search grid as v10.

In [7]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

param_dist_xgb = {
    'model__n_estimators'     : [500, 800, 1000, 1500, 2000],
    'model__max_depth'        : [4, 5, 6, 7, 8, 9],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.4, 0.5, 0.6, 0.7, 0.8],
    'model__min_child_weight' : [1, 3, 5, 7, 10],
    'model__reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'model__reg_lambda'       : [0.5, 1.0, 1.5, 2.0, 3.0],
}
xgb_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, tree_method='hist'))]),
    param_distributions=param_dist_xgb, n_iter=40, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
xgb_search.fit(X_train, y_train_log)
print(f'\nXGB best CV RMSE: ${-xgb_search.best_score_:,.0f}')
print('Best params:', xgb_search.best_params_)

XGB_PARAMS = {k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}
XGB_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':0, 'tree_method':'hist'})

xgb_check = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
xgb_check.fit(X_train, y_train_log)
xgb_val_rmse = rmse(y_val, np.expm1(xgb_check.predict(X_val)))
print(f'XGB val RMSE: ${xgb_val_rmse:,.0f}  (v10 was ~$21,730)')

Fitting 3 folds for each of 40 candidates, totalling 120 fits

XGB best CV RMSE: $22,644
Best params: {'model__subsample': 0.9, 'model__reg_lambda': 1.5, 'model__reg_alpha': 0.01, 'model__n_estimators': 2000, 'model__min_child_weight': 7, 'model__max_depth': 7, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.4}
XGB val RMSE: $21,679  (v10 was ~$21,730)


---
## 7. RandomizedSearchCV — LightGBM

In [8]:
param_dist_lgbm = {
    'model__n_estimators'     : [500, 800, 1000, 1500, 2000],
    'model__max_depth'        : [5, 6, 7, 8, 10, 12],
    'model__num_leaves'       : [60, 80, 100, 127, 200, 300],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.5, 0.6, 0.7, 0.8],
    'model__min_child_samples': [10, 20, 30, 50],
    'model__reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'model__reg_lambda'       : [0, 0.1, 0.5, 1.0],
}
lgbm_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1))]),
    param_distributions=param_dist_lgbm, n_iter=40, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
lgbm_search.fit(X_train, y_train_log)
print(f'\nLGBM best CV RMSE: ${-lgbm_search.best_score_:,.0f}')
print('Best params:', lgbm_search.best_params_)

LGBM_PARAMS = {k.replace('model__',''):v for k,v in lgbm_search.best_params_.items()}
LGBM_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':-1})

lgbm_check = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
lgbm_check.fit(X_train, y_train_log)
lgbm_val_rmse = rmse(y_val, np.expm1(lgbm_check.predict(X_val)))
print(f'LGBM val RMSE: ${lgbm_val_rmse:,.0f}')

Fitting 3 folds for each of 40 candidates, totalling 120 fits

LGBM best CV RMSE: $22,588
Best params: {'model__subsample': 0.8, 'model__reg_lambda': 0.5, 'model__reg_alpha': 0, 'model__num_leaves': 300, 'model__n_estimators': 1000, 'model__min_child_samples': 20, 'model__max_depth': 12, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.5}
LGBM val RMSE: $21,622


---
## 8. RandomizedSearchCV — Extra Trees

In [9]:
param_dist_et = {                                                                      
      'model__n_estimators'     : [200, 300, 500],                                                       
      'model__max_depth'        : [15, 20, 30],
      'model__min_samples_split': [2, 5, 10],                                                            
      'model__min_samples_leaf' : [1, 2, 4],                                                             
      'model__max_features'     : [0.5, 0.6, 0.7, 0.8],                                                  
  }                                                                                                      
et_search = RandomizedSearchCV(                                                        
      Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(random_state=42, n_jobs=-1))]),     
      param_distributions=param_dist_et, n_iter=15, cv=3,
      scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,                     
  )
et_search.fit(X_train, y_train_log)
print(f'ET best CV RMSE: ${-et_search.best_score_:,.0f}')
et_val_rmse = rmse(y_val, np.expm1(et_search.best_estimator_.predict(X_val)))
print(f'ET val RMSE:     ${et_val_rmse:,.0f}')

ET_PARAMS = {k.replace('model__',''):v for k,v in et_search.best_params_.items()}
ET_PARAMS.update({'random_state':42,'n_jobs':-1})
print(f'ET_PARAMS: {ET_PARAMS}')

Fitting 3 folds for each of 15 candidates, totalling 45 fits
ET best CV RMSE: $24,478
ET val RMSE:     $23,280
ET_PARAMS: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': 20, 'random_state': 42, 'n_jobs': -1}


---
## 9. Generate OOF Predictions (5-Fold) — 3 Models

Per-fold target encoding is **recomputed inside each fold** from fold-train rows only.
This covers `town`, `postal_sector`, `flat_model`, `h3_r8`, and `h3_r7`.

In [12]:
# Fixed Section 9 — OOF Predictions (5-Fold, 3 Models)
# Run after cells 1-8 of 11_h3_geospatial.ipynb

kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof  = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
et_oof   = np.zeros(len(X))

xgb_test_folds  = np.zeros((5, len(X_test)))
lgbm_test_folds = np.zeros((5, len(X_test)))
et_test_folds   = np.zeros((5, len(X_test)))

fold_rmses_xgb  = []
fold_rmses_lgbm = []
fold_rmses_et   = []

# (group_col, price_col, min_count)
# group_col exists in train_fe but NOT in X (h3_r7/h3_r8 were dropped from X)
ENCODE_PAIRS = [
    ('town',          'town_median_price',          1),
    ('postal_sector', 'postal_sector_median_price', MIN_COUNT_SECTOR),
    ('flat_model',    'flat_model_median_price',    1),
    ('h3_r8',         'h3_r8_median_price',         MIN_COUNT_H3_R8),
    ('h3_r7',         'h3_r7_median_price',         MIN_COUNT_H3_R7),
]

print('Generating OOF predictions (5-fold, 3 models)...')
print(f'{"Fold":>5}  {"XGB RMSE":>12}  {"LGBM RMSE":>12}  {"ET RMSE":>12}')
print('-' * 55)

for fold, (tr_idx, va_idx) in enumerate(kf5.split(X)):
    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr, y_va = y[tr_idx], y[va_idx]
    y_tr_log      = np.log1p(y_tr)
    global_med_tr = np.median(y_tr)

    # FIX: use train_fe (not X_tr) for group column lookups
    # train_fe still has h3_r7, h3_r8 — X does not
    fe_tr = train_fe.iloc[tr_idx]
    fe_va = train_fe.iloc[va_idx]

    for group_col, price_col, min_ct in ENCODE_PAIRS:
        fold_counts = {}
        fold_vals   = {}
        for g, p in zip(fe_tr[group_col].values, y_tr):
            fold_counts[g] = fold_counts.get(g, 0) + 1
            fold_vals.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_vals.items()
                    if fold_counts[g] >= min_ct}
        X_tr[price_col] = fe_tr[group_col].map(fold_map).fillna(global_med_tr).values
        X_va[price_col] = fe_va[group_col].map(fold_map).fillna(global_med_tr).values

    # XGBoost
    xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    xgb_pipe.fit(X_tr, y_tr_log)
    xgb_oof[va_idx]      = xgb_pipe.predict(X_va)
    xgb_test_folds[fold] = xgb_pipe.predict(X_test)
    fold_rmses_xgb.append(rmse(y_va, np.expm1(xgb_oof[va_idx])))

    # LightGBM
    lgbm_pipe = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    lgbm_pipe.fit(X_tr, y_tr_log)
    lgbm_oof[va_idx]      = lgbm_pipe.predict(X_va)
    lgbm_test_folds[fold] = lgbm_pipe.predict(X_test)
    fold_rmses_lgbm.append(rmse(y_va, np.expm1(lgbm_oof[va_idx])))

    # Extra Trees
    et_pipe = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**ET_PARAMS))])
    et_pipe.fit(X_tr, y_tr_log)
    et_oof[va_idx]      = et_pipe.predict(X_va)
    et_test_folds[fold] = et_pipe.predict(X_test)
    fold_rmses_et.append(rmse(y_va, np.expm1(et_oof[va_idx])))

    print(f'{fold+1:>5}  ${fold_rmses_xgb[-1]:>10,.0f}  ${fold_rmses_lgbm[-1]:>10,.0f}  ${fold_rmses_et[-1]:>10,.0f}')

print('-' * 55)
print(f'{"Mean":>5}  ${np.mean(fold_rmses_xgb):>10,.0f}  ${np.mean(fold_rmses_lgbm):>10,.0f}  ${np.mean(fold_rmses_et):>10,.0f}')

xgb_test_avg  = xgb_test_folds.mean(axis=0)
lgbm_test_avg = lgbm_test_folds.mean(axis=0)
et_test_avg   = et_test_folds.mean(axis=0)

print('\nOOF generation complete.')

Generating OOF predictions (5-fold, 3 models)...
 Fold      XGB RMSE     LGBM RMSE       ET RMSE
-------------------------------------------------------
    1  $    21,594  $    21,496  $    23,217
    2  $    22,347  $    22,359  $    23,973
    3  $    21,738  $    21,613  $    23,530
    4  $    21,873  $    21,880  $    23,682
    5  $    21,894  $    21,718  $    23,567
-------------------------------------------------------
 Mean  $    21,889  $    21,813  $    23,594

OOF generation complete.


---
## 10. Sanity Check — Individual Models & Blends

In [13]:
print('Individual OOF RMSE:')
print(f'  XGB:  ${rmse(y, np.expm1(xgb_oof)):>8,.0f}')
print(f'  LGBM: ${rmse(y, np.expm1(lgbm_oof)):>8,.0f}')
print(f'  ET:   ${rmse(y, np.expm1(et_oof)):>8,.0f}')

blend_equal = np.expm1((xgb_oof + lgbm_oof + et_oof) / 3)
print(f'\n3-model equal-weight blend OOF RMSE: ${rmse(y, blend_equal):>8,.0f}')
print(f'v10 OOF RMSE (for comparison):        ~$21,756')

Individual OOF RMSE:
  XGB:  $  21,890
  LGBM: $  21,815
  ET:   $  23,595

3-model equal-weight blend OOF RMSE: $  21,756
v10 OOF RMSE (for comparison):        ~$21,756


---
## 11. Ridge Meta-Model on 3 OOF Features

In [14]:
meta_X_train = np.column_stack([xgb_oof, lgbm_oof, et_oof])
meta_X_test  = np.column_stack([xgb_test_avg, lgbm_test_avg, et_test_avg])

print('Ridge alpha sweep (3 meta-features):')
print(f'{"alpha":>10}  {"RMSE":>12}  {"XGB coef":>10}  {"LGBM coef":>10}  {"ET coef":>10}')
print('-' * 62)

best_meta_rmse   = float('inf')
best_alpha_ridge = 1.0

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(meta_X_train, y_log)
    meta_pred_log = ridge.predict(meta_X_train)
    meta_rmse_val = rmse(y, np.expm1(meta_pred_log))
    print(f'{alpha:>10.3f}  ${meta_rmse_val:>10,.0f}  {ridge.coef_[0]:>10.4f}  {ridge.coef_[1]:>10.4f}  {ridge.coef_[2]:>10.4f}')
    if meta_rmse_val < best_meta_rmse:
        best_meta_rmse   = meta_rmse_val
        best_alpha_ridge = alpha

print(f'\nBest Ridge alpha: {best_alpha_ridge}')
meta_model = Ridge(alpha=best_alpha_ridge)
meta_model.fit(meta_X_train, y_log)
print(f'Learned coefficients:')
print(f'  XGB:  {meta_model.coef_[0]:.4f}')
print(f'  LGBM: {meta_model.coef_[1]:.4f}')
print(f'  ET:   {meta_model.coef_[2]:.4f}')
print(f'  Intercept: {meta_model.intercept_:.4f}')
print(f'\nOOF meta-train RMSE (optimistic): ${best_meta_rmse:,.0f}')
print(f'v10 Kaggle RMSE for comparison:    $21,755.56')

Ridge alpha sweep (3 meta-features):
     alpha          RMSE    XGB coef   LGBM coef     ET coef
--------------------------------------------------------------
     0.001  $    21,625      0.4173      0.4483      0.1368
     0.010  $    21,625      0.4173      0.4483      0.1368
     0.100  $    21,625      0.4174      0.4478      0.1372
     1.000  $    21,625      0.4179      0.4437      0.1408
    10.000  $    21,630      0.4136      0.4186      0.1702
   100.000  $    21,684      0.3693      0.3646      0.2670

Best Ridge alpha: 0.001
Learned coefficients:
  XGB:  0.4173
  LGBM: 0.4483
  ET:   0.1368
  Intercept: -0.0314

OOF meta-train RMSE (optimistic): $21,625
v10 Kaggle RMSE for comparison:    $21,755.56


---
## 12. Generate Submission v11

In [ ]:
final_log  = meta_model.predict(meta_X_test)
final_pred = np.expm1(final_log)

sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v11 = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred})
sub_v11 = sub_v11.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v11_h3_geo.csv'
sub_v11.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {sub_v11.shape}')
print(f'Price range: ${sub_v11.Predicted.min():,.0f} – ${sub_v11.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v11.Predicted.mean():,.0f}')

---
## 13. Summary

### Changes vs v10
| Change | Why | Expected Gain |
|---|---|---|
| `h3_r8_median_price` OOF | Equal-area neighbourhood cells (~990 zones) — true geographic price signal | ~100–300 |
| `h3_r7_median_price` OOF | Equal-area district cells (~140 zones) — stable coarse signal | ~50–150 |

### Score tracker
| Version | Key change | OOF RMSE | Kaggle |
|---|---|---|---|
| v9 | Wider grid + richer OOF encoding | ~$21,627 | $21,755.56 |
| v10 | postal_sector min_count smoothing | ~$21,627 | $21,755.56 |
| **v11** | **H3 geospatial cell encoding** | **_(run above)_** | **_(submit)_** |

### Key questions after running
- Did `h3_r8_median_price` or `h3_r7_median_price` reduce OOF RMSE vs v10? → confirms geospatial signal
- How many H3 r8 cells had >= 10 training rows? → check coverage vs postal_sector
- Are the new H3 columns in the top-15 feature importances?

### Next steps if v11 improves
- [ ] Try H3 resolution 9 (~0.1 km²) with higher min_count (e.g. 20) for even finer granularity
- [ ] Add Optuna Bayesian HPO for smarter hyperparameter search
- [ ] Explore CatBoost as 4th base model